# **Rolling Stock ETL**

### Data Fetching

In [23]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        query = f"SELECT * FROM {table_name};"
        
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

# --- Configuration (same as before) ---
HOST_IP = "100.95.110.69"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.95.110.69


C:\Users\win 11\AppData\Local\Temp\ipykernel_11224\66124713.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


✅ Fetched 7760 rows from 'extraction'


In [24]:
df = df_original.copy(deep=True)
df = df[(df['status_id'] == 1) & (df['dept_name'] == 'Rolling-Stock')][['filename', 'workorder_id', 'json_data']]

len(df)

1224

In [25]:
import pandas as pd

valid_json = df['json_data']
valid_json = valid_json[valid_json.apply(lambda x: isinstance(x, dict))]

all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))

['air_standup', 'airbag-pressure', 'airbag_pressure', 'approval', 'cardan_shaft', 'cceb', 'greasing_cardan_shaft', 'notification', 'stamping', 'technician', 'train_startup_test', 'tyre-pressure', 'tyre-wear', 'tyre_pressure', 'tyre_wear', 'water_ponding', 'work_order']


### Current Collector and Brush Earthing

In [26]:
df['cceb'] = df['json_data'].apply(
    lambda x: x.get('cceb') if isinstance(x, dict) else None
)

df['workorder_id'] = (
    df['workorder_id']
    .apply(lambda x: int(x) if pd.notnull(x) else None)
)

df[['filename', 'workorder_id', 'cceb']].head(1).to_dict(orient='records')

[{'filename': 'RS_PM_WEK_4000586856.pdf',
  'workorder_id': 4000586856,
  'cceb': {'train_no': '29',
   'location': 'BRICKFIELDS DEPOT',
   'frequency': 'WEEKLY',
   'date': None,
   'currect_collector': {'ca': '7 mm',
    'cb': '13mm',
    'cc': '12 mm',
    'cd': '12Mm',
    'ce': '13Mm',
    'cf': '13mm'},
   'earthing': {'e1': 'N/A', 'e2': 'N/A', 'e3': '12mm', 'e4': 'N/A'},
   'technician_detail': {'technician_id': '11515',
    'technician_date': '25/02/2024'},
   'supervisor_detail': {'supervisor_id': '7127',
    'supervisor_date': '25/02/2024'},
   'remarks': None}}]

In [27]:
import pandas as pd
import numpy as np

na_like_values = []

def is_na_like(val):
    if isinstance(val, (list, dict, np.ndarray)):
        return False
    try:
        if pd.isna(val):
            return True
    except Exception:
        pass
    val_str = str(val).strip().upper()
    return val_str in na_like_values

def find_na_keys(d):
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_na_like(v)]

df['na_keys'] = df['cceb'].apply(find_na_keys)

df_with_na = df[df['na_keys'].apply(lambda x: len(x) > 0)]

df_with_na[['filename', 'workorder_id', 'na_keys']].head(5)

,filename,workorder_id,na_keys
0,RS_PM_WEK_4000586856.pdf,4000586856,"[date, remarks]"
3,RS_PM_MTH_4000446287.pdf,4000446287,[remarks]
7,RS_PM_MTH_4000590580.pdf,4000290580,"[train_no, remarks]"
8,RS_PM_MTH_4000449255.pdf,4000449255,[remarks]
9,RS_PM_HYL_4000493210.pdf,4000493210,[remarks]


In [28]:
from collections import Counter

na_counter = Counter(k for keys in df['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

print(na_summary)

        key  na_count
1   remarks       929
0      date        87
2  train_no        33


In [29]:

import numpy as np
import re
import pandas as pd

pattern = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def clean_value(val):
    """Clean individual values (string, dict, etc.)."""
    if isinstance(val, str):
        return '' if pattern.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

# df['cceb'] = df['cceb'].apply(clean_value)

# df['cceb'] = df['cceb'].replace(np.nan, '', regex=True)

df['cceb'].head(3)

0    {'train_no': '29', 'location': 'BRICKFIELDS DE...
1    {'train_no': '25', 'location': 'BRICKFIELDS DE...
3    {'train_no': '27', 'location': 'BRICKFIELDS DE...
Name: cceb, dtype: object

In [30]:
def extract_leaf_keys(d, parent=''):
    keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                keys.extend(extract_leaf_keys(v, full_key))
            else:
                keys.append(full_key)
    return keys

df['cceb_leaf_keys'] = df['cceb'].apply(
    lambda x: extract_leaf_keys(x) if isinstance(x, dict) else []
)

unique_keys = sorted(set(k for sublist in df['cceb_leaf_keys'] for k in sublist))

for k in unique_keys:
    print(k)


currect_collector.ca
currect_collector.cb
currect_collector.cc
currect_collector.cd
currect_collector.ce
currect_collector.cf
date
earthing.e1
earthing.e2
earthing.e3
earthing.e4
frequency
location
remarks
supervisor_detail.supervisor_date
supervisor_detail.supervisor_id
technician_detail.technician_date
technician_detail.technician_id
train_no


In [31]:
import json
import pandas as pd

def flatten_with_descriptions(row):
    flat = {}

    def recurse(subdict, parent=''):
        if subdict is None:
            return

        if isinstance(subdict, str):
            try:
                subdict = json.loads(subdict)
            except json.JSONDecodeError:
                return

        if not isinstance(subdict, dict):
            return

        for k, v in subdict.items():
            if len(k) == 1 and k.isalpha():
                new_parent = parent
            else:
                new_parent = f"{parent}.{k}" if parent else k

            if isinstance(v, dict):
                desc = v.get('description')
                if desc:
                    desc_key = (
                        desc.lower()
                        .replace(' ', '_')
                        .replace('/', '_')
                        .replace('&', 'and')
                    )
                    for sub_k, sub_v in v.items():
                        if sub_k != 'description':
                            flat[f"{new_parent}.{desc_key}.{sub_k}"] = sub_v
                else:
                    recurse(v, new_parent)
            else:
                flat[new_parent] = v

    recurse(row)
    return flat

flattened_rows = [flatten_with_descriptions(r) for r in df['cceb'].fillna({})]

cceb_df = pd.DataFrame(flattened_rows)
cceb_df.index = df.index
cceb_df['workorder_id'] = df['workorder_id'].astype('Int64')
cceb_df['filename'] = df['filename']

cceb_df

,train_no,location,frequency,date,currect_collector.ca,currect_collector.cb,currect_collector.cc,currect_collector.cd,currect_collector.ce,currect_collector.cf,...,earthing.e2,earthing.e3,earthing.e4,technician_detail.technician_id,technician_detail.technician_date,supervisor_detail.supervisor_id,supervisor_detail.supervisor_date,remarks,workorder_id,filename
0,29,BRICKFIELDS DEPOT,WEEKLY,None,7 mm,13mm,12 mm,12Mm,13Mm,13mm,...,N/A,12mm,N/A,11515,25/02/2024,7127,25/02/2024,None,4000586856,RS_PM_WEK_4000586856.pdf
1,25,BRICKFIELDS DEPOT,MONTHLY,12/07/2022,NEW,7mm,10mm,9mm,7mm,NEW,...,7mm,8mm,6mm,7276,12/05/2022,7066,12/05/2022,,4000464193,RS_PM_MTH_4000464193.pdf
3,27,BRICKFIELDS DEPOT,MONTHLY,25/01/2022,8.6,10.5,9.5,11.0,9.5,11.0,...,7.0,10.0,9.5,7205,25/01/2022,7127,25/01/2022,None,4000446287,RS_PM_MTH_4000446287.pdf
4,22,BRICKFIELDS DEPOT,WEEKLY,09/10/2023,12.5,12.0,11.F,10-F,10-5,11.5,...,9.5,SF,NA,7205,09/10/2023,7192,09/10/2023,NA,4000558454,RS_PM_WEK_4000558454.pdf
5,27,BRICKFIELDS DEPOT,MONTHLY,17/05/2022,4.0,Change NEW,f-5,10.5,Change New,Chang new,...,11.5,12.5,NA,7205,17/05/2022,7127,17/05/2022,,4000464732,RS_PM_MTH_4000464732.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7516,25,BRICKFIELDS DEPOT,WEEKLY,13/07/2023,5mm,5mm,15mm,6mm,12mm,2mm,...,NEW,NEW,None,None,None,Stamp and signed,None,None,4000538576,RS_PM_WEK_4000538576.pdf
7517,21,BRICKFIELDS DEPOT,WEEKLY,15/10/2022,4.5,9.5,6.5,11.5,10,12,...,12.5,NA,8.5,7205,15/10/2022,7127,15/10/2022,None,4000491866,RS_PM_WEK_4000491866.pdf
7518,21,BRICKFIELDS DEPOT,YEARLY,09/04/2022,6.0,8.0,6.0,7.0,8.0,7,...,5.0,6.0,NA,11517,09/04/2022,6196,09/04/2022,6196,4000457606,RS_PM_YRL_4000457606.pdf
7519,23,BRICKFIELDS DEPOT,WEEKLY,28/02/2025,11.0,10.0,11.0,10.0,8.5,4.5,...,8.5,N/A,9.0,11517,28/02/2025,7203,28/02/2025,None,4000658422,RS_PM_WEK_4000658422.pdf


In [32]:
for i, col in enumerate(cceb_df.columns, start=1):
    print(f"{i:3d}. {col}")

    if col == 'workorder_id':
        continue

    valid_workorders = cceb_df.loc[cceb_df[col].notna(), 'workorder_id'].unique()

    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
        print("-" * 80)


  1. train_no
   Work Orders with data (1185): 4000586856, 4000464193, 4000446287, 4000558454, 4000464732, 4000457924, 4000449255, 4000493210, 4000453153, 4000453431, 4000449251, 4000487566, 4000460876, 4000616771, 4000443565, 4000458057, 4000457726, 4000629069, 4000446285, 4000445410, 4000443515, 4000449254, 4000453217, 4000561801, 4000548185, 4000462536, 4000629779, 4000554182, 4000621978, 4000554057, 4000492477, 4000680359, 4000462534, 4000497784, 4000496417, 4000490238, 4000464730, 4000467264, 4000467263, 4000501704, 4000476338, 4000475575, 4000476339, 4000475576, 4000489152, 4000511157, 4000473611, 4000470656, 4000479375, 4000469186, 4000473835, 4000470659, 4000509731, 4000506095, 4000505664, 4000503974, 4000493840, 4000478753, 4000523943, 4000518913, 4000480744, 4000511496, 4000495936, 4000519624, 4000553939, 4000584992, 4000538228, 4000539394, 4000550832, 4000521566, 4000538230, 4000544001, 4000541671, 4000541633, 4000519616, 4000555376, 4000553634, 4000553633, 4000552278, 40005

In [33]:
output_path = '../../output/rsd/rolling_stock.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    cceb_df.to_excel(writer, index=False, sheet_name='cceb')

print(f"✅ Exported successfully to '{output_path}' (replaced existing sheet)")

✅ Exported successfully to '../../output/rsd/rolling_stock.xlsx' (replaced existing sheet)
